In [1]:
import torch
import torch.nn as nn

In [2]:
#kaiming init hardcode
layer = nn.Linear(50, 10)
layer.weight.data *= 6**0.5
torch.zero_(layer.bias.data)

tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

In [3]:
#pytorch implementation
nn.init.kaiming_uniform_(layer.weight.data)
torch.zero_(layer.bias.data)

tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

In [4]:
def use_kai_uniform(module):
    if isinstance(module, nn.Linear):
        nn.init.kaiming_uniform_(module.weight.data)
        torch.zero_(module.bias.data)
        
model = nn.Sequential(nn.Linear(50, 40), nn.ReLU(), nn.Linear(40, 30), nn.ReLU(), nn.Linear(30, 1))
model.apply(use_kai_uniform)

Sequential(
  (0): Linear(in_features=50, out_features=40, bias=True)
  (1): ReLU()
  (2): Linear(in_features=40, out_features=30, bias=True)
  (3): ReLU()
  (4): Linear(in_features=30, out_features=1, bias=True)
)

### leaky ReLU

In [5]:
alpha = 0.2
model = nn.Sequential(nn.Linear(50, 1), nn.LeakyReLU(negative_slope=alpha)).to('cuda')
nn.init.kaiming_normal_(model[0].weight, alpha, nonlinearity='leaky_relu')

Parameter containing:
tensor([[ 0.0191, -0.1795, -0.1598, -0.3625, -0.1174, -0.0455, -0.2429, -0.0395,
         -0.0472,  0.0270, -0.1973,  0.2645,  0.1166,  0.2946,  0.4660, -0.1057,
          0.1258, -0.2117, -0.0326,  0.0127,  0.4601,  0.1190,  0.2377,  0.0053,
          0.0968, -0.1857, -0.3832, -0.1828,  0.2828, -0.0635, -0.0768, -0.3499,
          0.1965, -0.0892, -0.0723, -0.2375,  0.0769,  0.4005,  0.2525,  0.2999,
         -0.1173,  0.0578, -0.1955,  0.2378,  0.0566, -0.1495,  0.0244,  0.0075,
         -0.3274,  0.1461]], device='cuda:0', requires_grad=True)

### Batch Norm

In [6]:
model = nn.Sequential(nn.Flatten(), nn.BatchNorm1d(28*28), nn.Linear(28*28, 300), nn.LeakyReLU(alpha),
                      nn.BatchNorm1d(300), nn.Linear(300, 100), nn.LeakyReLU(alpha),
                      nn.BatchNorm1d(100), nn.Linear(100, 10))
#then it is possible to train this model as earlier

In [7]:
model = nn.Sequential(nn.Flatten(), nn.Linear(28*28, 300, bias=False), nn.BatchNorm1d(300), nn.LeakyReLU(alpha),
                      nn.Linear(300, 100, bias=False), nn.BatchNorm1d(100), nn.LeakyReLU(alpha),
                      nn.Linear(100, 10))

### Layer Norm and Gradient clipping

In [8]:
inputs = torch.randn([32, 3, 200, 300])
layer_norm = nn.LayerNorm([200, 300])
result = layer_norm(inputs)
result

tensor([[[[-7.4875e-02, -4.9711e-01,  2.3918e+00,  ...,  5.1622e-01,
           -4.4308e-01, -3.5528e-01],
          [ 8.6355e-01,  2.9316e-01,  1.3020e+00,  ...,  6.5757e-01,
           -8.4269e-01, -2.3533e+00],
          [ 4.7054e-01,  7.6356e-01,  4.4432e-01,  ..., -1.2951e+00,
            1.2469e+00,  1.1738e+00],
          ...,
          [-7.3716e-01,  3.1498e-02,  1.7397e+00,  ..., -1.4093e+00,
           -1.6652e+00, -3.1202e-01],
          [ 3.6356e-03,  1.7363e+00, -1.2596e+00,  ..., -3.6789e-01,
            7.1518e-01,  2.0586e+00],
          [ 6.3133e-01,  3.0848e-01, -3.6211e-02,  ...,  4.3303e-01,
           -7.5919e-01, -4.7732e-01]],

         [[ 2.6395e-01,  2.2504e-01, -7.8695e-01,  ..., -1.3638e+00,
            2.4718e+00,  7.3111e-01],
          [ 1.4697e+00,  6.6772e-01, -1.5195e+00,  ..., -5.6929e-01,
            1.9139e-01,  1.0896e+00],
          [-2.5575e-01, -4.1195e-02,  1.2814e+00,  ...,  1.2573e-01,
           -2.1912e+00, -4.4165e-01],
          ...,
     

In [9]:
def train_model(model, optimizer, criterion, dataloader, n_epochs):
    model.train()
    for epoch in n_epochs():
        total_loss = 0
        for X_batch, y_batch in dataloader:
                X_batch, y_batch = X_batch.to('cuda'), y_batch.to('cuda')
                y_pred = model(X_batch)
                
                loss = criterion(y_pred, y_batch)
                total_loss += loss.item()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1) #<<----
                
                optimizer.step()
                optimizer.zero_grad()
                
        mean_loss = total_loss / len(dataloader)
        print(f"epoch: {epoch}, loss: {mean_loss}")
                

#### Pretrained Layers

In [10]:
torch.manual_seed(52)

model_A = nn.Sequential(nn.Flatten(), nn.Linear(28*28, 100), nn.ReLU(),
                        nn.Linear(100, 100), nn.ReLU(),
                        nn.Linear(100, 8))

###training this model

In [11]:
import copy

torch.manual_seed(52)

reused_layers = copy.deepcopy(model_A[:-1])
model_B_on_A = nn.Sequential(*reused_layers, nn.Linear(100, 1)).to('cuda')

In [12]:
for layer in model_B_on_A[:-1]:
    for param in layer.parameters():
        param.requires_grad = False

In [13]:
import torchmetrics

xentropy = nn.BCEWithLogitsLoss()
accuracy = torchmetrics.Accuracy(task='binary').to('cuda')

#  [trainning model_B_on_A]

In [14]:
my_int = 32
type(my_int)

int

In [15]:
mystr = '123.5'
my_int = int(float(mystr)) +1
my_int

124

In [16]:
def x_position(x0, v, t): 
    g = -9.8
    x1 = 1/2*g*t**2 + v*t + x0
    return x1

x_position(1, 10, 3)


-13.100000000000001

### Faster Optimizers

In [17]:
#momemntum
optimizer = torch.optim.SGD(params=model.parameters(), lr= 0.01, momentum=0.9)

In [18]:
#NAG
optimizer = torch.optim.SGD(params=model.parameters(), lr=0.02, momentum=0.9, nesterov=True)

In [19]:
#RMSProp
optimizer = torch.optim.RMSprop(params=model.parameters(), lr=0.05, alpha=0.9)

In [20]:
#Adam
optimizer = torch.optim.Adam(params=model.parameters(), lr=0.001, betas=(0.9, 0.999))

### Learning schedules 

In [21]:
#Exponential scheduling
optimizer = torch.optim.SGD(params=model.parameters(), lr=0.2)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer=optimizer, gamma=0.9)

def train_with_scheduling(model, optimizer, scheduler, criterion, dataloader, n_epochs):
    model.train()
    for epoch in n_epochs():
        total_loss = 0
        for X_batch, y_batch in dataloader:
                X_batch, y_batch = X_batch.to('cuda'), y_batch.to('cuda')
                y_pred = model(X_batch)
                
                loss = criterion(y_pred, y_batch)
                total_loss += loss.item()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1) #<<----
                
                optimizer.step()
                optimizer.zero_grad()
                scheduler.step()
                
        mean_loss = total_loss / len(dataloader)
        print(f"epoch: {epoch}, loss: {mean_loss}")

In [22]:
#cosine annealing
cosine_annealing = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer=optimizer, T_max=20, eta_min=0.001)

In [23]:
def evaluate_model(model, data_loader, metrics):
    model.eval()
    metrics.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to('cuda'), y_batch.to('cuda')
            y_pred = model(y_batch)
            metrics.update(y_pred, y_batch)
            
    return metrics.compute()

In [24]:
#Performance scheduling
performance_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer, mode='max', factor=0.1, patience=3)

def train_with_PerformanceScheduling(model, optimizer, scheduler, criterion, dataloader, valid_loader, metric, n_epochs):
    model.train()
    for epoch in n_epochs():
        total_loss = 0
        for X_batch, y_batch in dataloader:
                X_batch, y_batch = X_batch.to('cuda'), y_batch.to('cuda')
                y_pred = model(X_batch)
                
                loss = criterion(y_pred, y_batch)
                total_loss += loss.item()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1) #<<----
                
                optimizer.step()
                optimizer.zero_grad()
                
                val_score = evaluate_model(model, valid_loader, metric).item()
                scheduler.step(val_score)
                
        mean_loss = total_loss / len(dataloader)
        print(f"epoch: {epoch}, loss: {mean_loss}")

#### Warming up lr

In [25]:
warming_lr = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, end_factor=1, total_iters=3)

In [26]:
#the same
warming_lr = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda epoch: (min(epoch, 3)/3) * (1-0.1) + 0.1)

In [27]:
def train_with_warmup(model, optimizer, scheduler, criterion, dataloader, valid_loader, metric, warmup_scheduler, n_epochs):
    model.train()
    for epoch in n_epochs():
        total_loss = 0
        warmup_scheduler.step()
        for X_batch, y_batch in dataloader:
                X_batch, y_batch = X_batch.to('cuda'), y_batch.to('cuda')
                y_pred = model(X_batch)
                
                loss = criterion(y_pred, y_batch)
                total_loss += loss.item()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1) #<<----
                
                optimizer.step()
                optimizer.zero_grad()
                
                if epoch>=3:
                    val_score = evaluate_model(model, valid_loader, metric).item()
                    scheduler.step(val_score)
                
        mean_loss = total_loss / len(dataloader)
        print(f"epoch: {epoch}, loss: {mean_loss}")

In [28]:
## Cosine repeat scheduler
cosine_repeat = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=2, T_mult=2, eta_min=0.001)

## Regularization

In [29]:
#l2 for every param
optimizer = torch.optim.SGD(model.parameters(), lr=0.02, weight_decay=1e-4)

In [30]:
#l2 for selected params
def train_l2(model, optimizer, criterion, dataloader, n_epochs):
    model.train()
    
    params_to_regularize = [param for name, param in model.named_parameters()
                            if not 'bias' in name and not 'bn' in name]
    
    for epoch in n_epochs():
        total_loss = 0
        for X_batch, y_batch in dataloader:
                X_batch, y_batch = X_batch.to('cuda'), y_batch.to('cuda')
                y_pred = model(X_batch)
                
                loss = criterion(y_pred, y_batch)
                l2_loss = sum(param.pow(2).sum() for param in params_to_regularize)
                total_loss += loss.item() + 1e-4 * l2_loss
                loss.backward()
                
                optimizer.step()
                optimizer.zero_grad()
                
        mean_loss = total_loss / len(dataloader)
        print(f"epoch: {epoch}, loss: {mean_loss}")

In [31]:
#or it is possible to initialize an optimizer with only parameteres needed to be regularized
params_bias_and_bn = [param for name, param in model.named_parameters()
                      if 'bias' in name or 'bn' in name]

params_to_regularize = [param for name, param in model.named_parameters()
                            if not 'bias' in name and not 'bn' in name]

optimizer = torch.optim.SGD([{"params": params_to_regularize, "weight_decay": 1e-4},
                             {"params": params_bias_and_bn}], lr=0.05)

In [32]:
#l1
#...[training function]...
total_loss = 0.02 # for example
l1_loss = sum(param.abs().sum() for param in params_to_regularize)
loss = total_loss + l1_loss * 1e-4

In [33]:
## Dropout

model = nn.Sequential(nn.Flatten(), nn.Dropout(0.2), nn.Linear(28*28, 300), nn.ReLU(),
                      nn.Dropout(0.2), nn.Linear(300, 200), nn.ReLU(),
                      nn.Dropout(0.2), nn.Linear(200, 100), nn.ReLU(),
                      nn.Dropout(0.2), nn.Linear(100, 10)).to('cuda')

In [34]:
torch.manual_seed(52)
model.eval()
for module in model.modules():
    if isinstance(module, nn.Dropout):
        module.train()
        
X_new = torch.randn((3, 1, 28, 28)).to('cuda') #pretend that there is some useful data

with torch.no_grad():
    X_repeated = X_new.repeat_interleave(100, dim=0).to('cuda')
    y_logits = model(X_repeated).reshape(3, 100, 10)
    y_probas = nn.functional.softmax(y_logits, dim=-1)
    y_probas = y_probas.mean(dim=1)

y_probas.round(decimals=2)

tensor([[0.1100, 0.1100, 0.1000, 0.1000, 0.0900, 0.0900, 0.1000, 0.1100, 0.1100,
         0.0900],
        [0.1100, 0.1100, 0.1000, 0.1000, 0.0900, 0.1000, 0.1000, 0.1000, 0.1100,
         0.1000],
        [0.1100, 0.1100, 0.1000, 0.1000, 0.0900, 0.0900, 0.1000, 0.1000, 0.1100,
         0.0900]], device='cuda:0')

In [ ]:
#dropout monte carlo
import torch.nn.functional as F

class MCDropout(nn.Dropout):
    def forward(self, input):
        return F.dropout(input, self.p, training=True)

In [37]:
# Max Norm Regularization
def apply_max_norm(model, max_norm=2, epsilon=1e-8, dim=1):
    with torch.no_grad():
        for name, param in model.named_parameters():
            if 'bias' not in name:
                actual_norm = param.norm(p=2, dim=dim, keepdim=True)
                target_norm = torch.clamp(actual_norm, 0, max_norm)
                param *= target_norm / (epsilon + actual_norm)